In [1]:
import pandas as pd

In [2]:
import pandas as pd

season = "2627"
praefix = season[:2]  
leagues = {
    "pl": "E0",  
    "de": "D1",  
    "es": "SP1",  
    "it": "I1",  
    "fr": "F1", 
}
urls = {
    name: f"https://football-data.co.uk/mmz4281/{season}/{code}.csv"
    for name, code in leagues.items()
}

In [3]:
import numpy as np


In [4]:
def snake_case(df):
    df = df.copy()
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
    )
    return df


In [5]:
column_mapping = {
    "hometeam": "home_team",
    "awayteam": "away_team",
    "fthg": "full_time_home_goals",
    "ftag": "full_time_away_goals",
    "ftr": "full_time_result",
    "hthg": "half_time_home_goals",
    "htag": "half_time_away_goals",
    "htr": "half_time_result",
    "hs": "home_shots",
    "as": "away_shots",
    "hst": "home_shots_on_target",
    "ast": "away_shots_on_target"
}



In [6]:
columns_for_standings = [
    "date",
    "home_team",
    "away_team",
    "full_time_home_goals",
    "full_time_away_goals",
    "full_time_result",
]

In [7]:
league_data = {}

for name, code in leagues.items():
    league_data[name] = {}

    full = pd.read_csv(urls[name], encoding="utf-8-sig")
    full = snake_case(full)
    full = full.rename(columns=column_mapping)

    league_data[name]["full"] = full
    league_data[name]["matches"] = full[columns_for_standings].copy()

    matches = league_data[name]["matches"]

    # Combine home and away teams, remove duplicates, then sort them.
    league_data[name]["teams"] = sorted(
        set(matches["home_team"]) | set(matches["away_team"])
    )

    league_data[name]["team_matches"] = {}

    for team in league_data[name]["teams"]:
        team_matches = matches[
            (matches["home_team"] == team)
            | (matches["away_team"] == team)
        ].copy()

        league_data[name]["team_matches"][team] = team_matches


In [8]:
for name in leagues:
    standings_rows = []

    for team in league_data[name]["teams"]:
        club_df = league_data[name]["team_matches"][team]

        
        played = club_df.dropna(
            subset=["full_time_home_goals", "full_time_away_goals"]
        )

        home_games = played[played["home_team"] == team]
        away_games = played[played["away_team"] == team]

        home_wins = (
            home_games["full_time_home_goals"]
            > home_games["full_time_away_goals"]
        ).sum()
        away_wins = (
            away_games["full_time_away_goals"]
            > away_games["full_time_home_goals"]
        ).sum()
        wins = home_wins + away_wins

        home_draws = (
            home_games["full_time_home_goals"]
            == home_games["full_time_away_goals"]
        ).sum()
        away_draws = (
            away_games["full_time_away_goals"]
            == away_games["full_time_home_goals"]
        ).sum()
        draws = home_draws + away_draws

        home_losses = (
            home_games["full_time_home_goals"]
            < home_games["full_time_away_goals"]
        ).sum()
        away_losses = (
            away_games["full_time_away_goals"]
            < away_games["full_time_home_goals"]
        ).sum()
        losses = home_losses + away_losses

        goals_for = (
            home_games["full_time_home_goals"].sum()
            + away_games["full_time_away_goals"].sum()
        )
        goals_against = (
            home_games["full_time_away_goals"].sum()
            + away_games["full_time_home_goals"].sum()
        )
        goal_difference = goals_for - goals_against
        points = 3 * wins + draws

        standings_rows.append({
            "Team": team,
            "Played": len(played),
            "W": wins,
            "D": draws,
            "L": losses,
            "GF": goals_for,
            "GA": goals_against,
            "GD": goal_difference,
            "Points": points,
        })

    league_data[name]["standings"] = (
        pd.DataFrame(standings_rows)
        .sort_values(
            ["Points", "GD", "GF"],
            ascending=False,
        )
        .reset_index(drop=True)
    )


In [9]:
file_path = "football_standings.xlsx"

with pd.ExcelWriter(file_path, engine="openpyxl") as writer:
    for name in leagues:
        table = league_data[name]["standings"].copy()
        table.index = range(1, len(table) + 1)

        table.to_excel(
            writer,
            sheet_name=name.upper(),
            index_label="Pos"
        )